In [1]:
import pynvml
import os
pynvml.nvmlInit()
num_gpus = pynvml.nvmlDeviceGetCount()
selected_gpus = []

for i in range(num_gpus):
    handle = pynvml.nvmlDeviceGetHandleByIndex(i)
    meminfo = pynvml.nvmlDeviceGetMemoryInfo(handle)
    used_percent = meminfo.used / meminfo.total * 100
    print(f"GPU {i}: used {used_percent:.1f}% of {meminfo.total/1024**3:.2f} GB")
    if used_percent < 50:
        selected_gpus.append(i)
pynvml.nvmlShutdown()

if not selected_gpus:
    raise RuntimeError("No GPUs with memory usage less than 50% found!")
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(i) for i in selected_gpus)
print("Selected GPUs:", os.environ["CUDA_VISIBLE_DEVICES"])

GPU 0: used 3.9% of 8.00 GB
Selected GPUs: 0


In [2]:
import re
import json
import torch
import itertools
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, HfArgumentParser
from trl import DPOTrainer, DPOConfig
from typing import Dict, Optional
from huggingface_hub import HfApi, create_repo, login

In [3]:
print("Available GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

Available GPUs: 1
GPU 0: NVIDIA GeForce RTX 3070 Laptop GPU


In [4]:
torch.cuda.empty_cache()

# Task 1 - Finding a suitable dataset

In [ ]:
def preprocess_sample(sample: dict) -> dict:
    # Extract the prompt, chosen response, and rejected response directly
    prompt = sample["prompt"].strip()
    chosen_response = sample["chosen"].strip()
    rejected_response = sample["rejected"].strip()

    # Return the processed sample
    return {
        "prompt": prompt,
        "chosen": chosen_response,
        "rejected": rejected_response,
    }

def get_hh(split: str, sanity_check: bool = False, cache_dir: str = None) -> Dataset:
    # Load the dataset
    dataset = load_dataset("Dahoas/rm-static", split=split, cache_dir=cache_dir)

    # Limit the dataset size to 5 due to computational constraints
    if sanity_check:
        dataset = dataset.select(range(min(len(dataset), 5)))

    # Apply the preprocessing function
    return dataset.map(preprocess_sample)

In [7]:
sanity_check = True
train_dataset = get_hh("train", sanity_check=sanity_check)
eval_dataset  = get_hh("test", sanity_check=sanity_check)

In [8]:
print("Prompt:", train_dataset["prompt"][1])
print("Chosen Response:", train_dataset["chosen"][1])
print("Rejected Response:", train_dataset["rejected"][1])

Prompt: Human: What are some foods that are good for diabetics?

Assistant: To be honest, some of these are better than others, and they’re a little more like opinions than facts. For example, many of the diets say to limit vegetables with high sugar content, and there’s some debate on the subject, as far as how much of these vegetables are actually bad for diabetics.

Human: Okay, any other advice?

Assistant:
Chosen Response: What exactly are you asking? There’s a lot of different kinds of diabetic diets. I could try to recommend you some specific foods and recipes. I could help you look up any of the foods, and I could find recipes for them.
Rejected Response: Sure, we’ve got information on common mistakes that diabetic patients make with their diets, and even some specific things to do when you eat out and on the go.  One thing that’s recommended in these articles is just to be very mindful of the timing of food intake.


# Task 2 - Training a model with DPOTrainer

In [9]:
model_name_or_path = "Qwen/Qwen2-0.5B-Instruct"
ignore_bias_buffers = False

model = AutoModelForCausalLM.from_pretrained(model_name_or_path)
if ignore_bias_buffers:
    # torch distributed hack
    model._ddp_params_and_buffers_to_ignore = [
        name for name, buffer in model.named_buffers() if buffer.dtype == torch.bool
    ]

ref_model = AutoModelForCausalLM.from_pretrained(model_name_or_path)
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


In [10]:
# Move models to CUDA (DeepSpeed will handle distribution across selected GPUs)
device = torch.device("cuda")
model.to(device)
ref_model.to(device)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2RotaryEmbe

In [11]:
learning_rates = [3e-5]  # Example learning rates
batch_sizes = [4]          # Example batch sizes
num_epochs = [3]           # Example number of epochs
betas = [0.1]            # Example beta values

In [12]:
hyperparameter_combinations = list(itertools.product(learning_rates, batch_sizes, num_epochs, betas))

In [14]:
# Step 4: Hyperparameter search loop
results = []
best_loss = float("inf")  # Initialize best loss as infinity
best_model_path = None

In [15]:
for lr, batch_size, epochs, beta in hyperparameter_combinations:
    print(f"\nTraining with lr={lr}, batch_size={batch_size}, epochs={epochs}, beta={beta}")
    output_dir = f"./dpo_lr{lr}_bs{batch_size}_ep{epochs}_beta{beta}"

    # Configure DPO training
    dpo_config = DPOConfig(
        output_dir=output_dir,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        logging_dir="./logs",
        logging_steps=10,
        save_total_limit=2,
        learning_rate=lr,
        report_to="none",
        beta=beta,  # Temperature parameter for preference weighting
    )

    # Initialize DPOTrainer
    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=ref_model,
        args=dpo_config,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
    )

    # Train the model
    dpo_trainer.train()

    # Evaluate the model
    eval_results = dpo_trainer.evaluate()
    loss = eval_results.get("eval_loss", None)
    results.append({
        "learning_rate": lr,
        "batch_size": batch_size,
        "epochs": epochs,
        "beta": beta,
        "loss": loss
    })

    # Track the best model
    if loss is not None and loss < best_loss:
        best_loss = loss
        best_model_path = output_dir
        print(f"New best model found! Saving model at: {best_model_path}")


Training with lr=3e-05, batch_size=4, epochs=3, beta=0.1


C:\Users\ws-\AppData\Roaming\Python\Python312\site-packages\transformers\training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
1,No log,0.613500,-0.275840,-0.479901,0.375000,0.204061,-98.468109,-115.291000,-3.154367,-2.806827
2,No log,0.623796,-0.735461,-0.904884,0.250000,0.169423,-103.064323,-119.540840,-3.242298,-2.859205
3,No log,0.631183,-0.912344,-1.066884,0.250000,0.154541,-104.833153,-121.160835,-3.263175,-2.876090


New best model found! Saving model at: ./dpo_lr3e-05_bs4_ep3_beta0.1


In [16]:
# Save the model and tokenizer locally.
model.save_pretrained("./dpo_finetuned_qwen_model")
tokenizer.save_pretrained("./dpo_finetuned_qwen_model")

('./dpo_finetuned_qwen_model\\tokenizer_config.json',
 './dpo_finetuned_qwen_model\\special_tokens_map.json',
 './dpo_finetuned_qwen_model\\vocab.json',
 './dpo_finetuned_qwen_model\\merges.txt',
 './dpo_finetuned_qwen_model\\added_tokens.json',
 './dpo_finetuned_qwen_model\\tokenizer.json')

In [17]:
model = AutoModelForCausalLM.from_pretrained("./dpo_finetuned_qwen_model")
tokenizer = AutoTokenizer.from_pretrained("./dpo_finetuned_qwen_model")

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


In [ ]:
def generate_response(prompt: str, max_length: int = 250) -> str:
    # Tokenize the input prompt
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    
    # Generate the response
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_length=max_length,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            num_return_sequences=1,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode the generated output
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Remove the prompt from the generated text
    response = full_text[len(prompt):].strip()
    
    return response

In [19]:
# Example single inference:
sample_prompt = "What will the AI be in next 10 years?"
print("\nInference Example:")
print("\nPrompt:", sample_prompt)
print("\nResponse:", generate_response(sample_prompt, max_length=250))


Inference Example:

Prompt: What will the AI be in next 10 years?

Response: What kind of AI would you like to know? I could try to find some AI-related topics and answer your question.


# Task 3 - Pushing the Model to Hugging Face Hub

In [ ]:
# hugging face login delete due to git hub regulations

In [ ]:
repo_id = 'st125338/npu_a5_dpo_qwen2_model'
create_repo(repo_id, repo_type='model', private=False)

# Push the dataset to Hugging Face
model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

model.safetensors: 100%|██████████| 1.98G/1.98G [01:40<00:00, 19.7MB/s]


CommitInfo(commit_url='https://huggingface.co/st125338/npu_a5_dpo_qwen2_model/commit/b280105cfccffe14ca932151f278e568a55d0180', commit_message='Upload Qwen2ForCausalLM', commit_description='', oid='b280105cfccffe14ca932151f278e568a55d0180', pr_url=None, repo_url=RepoUrl('https://huggingface.co/st125338/npu_a5_dpo_qwen2_model', endpoint='https://huggingface.co', repo_type='model', repo_id='st125338/npu_a5_dpo_qwen2_model'), pr_revision=None, pr_num=None)